# Solutions

:::{admonition} Reference solutions
:class: note
Worked solutions for the short exercises in [bonus-b-defensive-programming-and-packaging-exercises.ipynb](bonus-b-defensive-programming-and-packaging-exercises.ipynb).
:::

## Exercise 1: An invariant with assert

Write `normalise(weights)` that divides each weight by the total, and asserts the invariant that the result sums to 1 (within a small tolerance). Test it on `[2.0, 2.0]`.

In [ ]:
def normalise(weights):
    total = sum(weights)
    out = [w / total for w in weights]
    assert abs(sum(out) - 1.0) < 1e-9, "weights must sum to 1"
    return out

print(normalise([2.0, 2.0]))   # [0.5, 0.5]

## Exercise 2: A custom exception

Define `NegativeDischargeError(ValueError)` and a function `check(q_m3s)` that raises it when discharge is negative. Catch it and print the message.

In [ ]:
class NegativeDischargeError(ValueError):
    pass

def check(q_m3s):
    if q_m3s < 0.0:
        raise NegativeDischargeError(f"discharge {q_m3s} < 0")
    return q_m3s

try:
    check(-3.0)
except NegativeDischargeError as err:
    print("caught:", err)

## Exercise 3: Try / except / else / finally

Write `safe_divide(a, b)` that returns `a / b`, catches `ZeroDivisionError` (returning `None`), prints a message in the `else` branch when it succeeds, and always prints "done" in `finally`. Call it with `(6, 2)` and `(6, 0)`.

In [ ]:
def safe_divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError:
        print("cannot divide by zero")
        return None
    else:
        print("division ok")
        return result
    finally:
        print("done")

print(safe_divide(6, 2))
print(safe_divide(6, 0))

## Exercise 4: Validate preconditions

Write `validate(temp_celsius, rh_percent)` that raises `ValueError` if the temperature is below absolute zero or the relative humidity is outside 0–100 %. Show it passing on valid input and raising on `rh_percent = 150`.

In [ ]:
def validate(temp_celsius, rh_percent):
    if temp_celsius < -273.15:
        raise ValueError("temperature below absolute zero")
    if not (0.0 <= rh_percent <= 100.0):
        raise ValueError("relative humidity out of range")

validate(20.0, 55.0)
try:
    validate(20.0, 150.0)
except ValueError as err:
    print("caught:", err)

## Exercise 5: Logging with levels

Configure logging to stdout at INFO level, get a logger, and emit a debug message (which should be suppressed), an info message, and a warning.

In [ ]:
import logging
import sys
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s",
                    stream=sys.stdout, force=True)
log = logging.getLogger("demo")
log.debug("suppressed")
log.info("started")
log.warning("low sample count")

## Exercise 6: Write and run a pytest suite

Write a module `mymod.py` with `double(x)` returning `2 * x`, and a parametrised test file that checks three cases. Run pytest on it with a subprocess and print the output.

In [ ]:
from pathlib import Path
import subprocess
import sys
Path("_files").mkdir(exist_ok=True)
Path("_files/mymod.py").write_text("def double(x):\n    return 2 * x\n", encoding="utf-8")
Path("_files/test_mymod.py").write_text(
    "import pytest\n"
    "from mymod import double\n"
    "@pytest.mark.parametrize('x, y', [(1, 2), (3, 6), (0, 0)])\n"
    "def test_double(x, y):\n"
    "    assert double(x) == y\n",
    encoding="utf-8",
)
result = subprocess.run([sys.executable, "-m", "pytest", "test_mymod.py", "-q"],
                        capture_output=True, text=True, cwd="_files")
print(result.stdout.strip())

## Exercise 7: Replace assert-based validation

The line `assert rh_percent >= 0, "negative humidity"` disappears under `python -O`. Rewrite the check as a function that raises `ValueError`, so it fires regardless of optimisation. Demonstrate on a valid value.

In [ ]:
def validate_humidity(rh_percent):
    if rh_percent < 0.0:
        raise ValueError("negative humidity")
    return rh_percent

print(validate_humidity(55.0))